# Heat Exchanger Fouling Detection Suite

**Kern-Seaton model · Physics-inspired prediction · Early anomaly detection · Health index · RUL · Cleaning economics**

Flat project: all **code**, CSVs and diagrams in one folder. Run all cells top to bottom.


## Project map (Mermaid)

### Pipeline

```mermaid
flowchart LR
    Sensors --> KernSeaton[Kern-Seaton Rf]
    KernSeaton --> Predictor
    Predictor --> Anomalies
    Predictor --> HealthIndex
    Predictor --> RUL
    Anomalies --> Alerts
    HealthIndex --> Alerts
    RUL --> CleaningPlan
    CleaningPlan --> CostOptimization
```

### Severity states

```mermaid
stateDiagram-v2
    [*] --> Clean
    Clean --> Mild: Rf rising
    Mild --> Elevated: Rf > 70% critical
    Elevated --> NearCritical: Rf > 90% critical
    NearCritical --> Critical: Rf ≥ Rf_CRIT
    Critical --> Clean: Clean executed
    Mild --> Clean: Early intervention
```

### Health Index weights

```mermaid
pie title Health Index
    "Rf 40%" : 40
    "ΔP 25%" : 25
    "Efficiency 25%" : 25
    "Anomaly 10%" : 10
```


## 0. Imports & configuration


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams.update({
    'figure.figsize': (10, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold'
})

Rf_CRIT = 1.8e-4   # m2K/W -- schedule cleaning
dP_CRIT = 55.0     # kPa

print('Libraries loaded.')
print('Critical Rf = %.1e m2K/W | Critical dP = %.1f kPa' % (Rf_CRIT, dP_CRIT))


## 1. Kern-Seaton asymptotic fouling model

$$
R_f(t) = R_f^{\infty}\left(1 - e^{-t/\tau_f}\right)
$$

Asymptotic resistance and time constant depend on water hardness, temperature and velocity.


In [ ]:
def kern_seaton_Rf(hours, hardness, T_hot, velocity):
    '''Asymptotic fouling resistance [m2K/W].'''
    hours = np.asarray(hours, dtype=float)
    hardness = np.asarray(hardness, dtype=float)
    T_hot = np.asarray(T_hot, dtype=float)
    velocity = np.asarray(velocity, dtype=float)
    Rf_inf = 1.2e-4 * (hardness / 200.0) ** 1.1 * np.exp(0.012 * (T_hot - 100.0))
    tau_f = 8000.0 * (velocity / 1.5) ** 0.8 / (hardness / 200.0) ** 0.3
    return np.clip(Rf_inf * (1.0 - np.exp(-hours / np.maximum(tau_f, 1.0))), 0, None)


t = np.linspace(0, 30000, 250)
fig, ax = plt.subplots(figsize=(9, 4.5))
for label, hard, T, vel in [
    ('Mild (100 ppm, 120 C, 2.0 m/s)', 100, 120, 2.0),
    ('Moderate (300 ppm, 150 C, 1.5 m/s)', 300, 150, 1.5),
    ('Severe (500 ppm, 170 C, 1.0 m/s)', 500, 170, 1.0),
]:
    ax.plot(t / 1000, kern_seaton_Rf(t, hard, T, vel) * 1e4, lw=2, label=label)
ax.axhline(Rf_CRIT * 1e4, color='r', ls='--', label='Critical Rf')
ax.set_xlabel('Operating time [x1000 h]')
ax.set_ylabel('Rf x 10^4 [m2K/W]')
ax.set_title('Kern-Seaton Asymptotic Fouling Model')
ax.legend()
plt.tight_layout()
plt.savefig('07_kern_seaton_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 07_kern_seaton_curves.png')


## 2. 3D fouling surface (hardness x time)


In [ ]:
H = np.linspace(100, 500, 40)
T = np.linspace(0, 40000, 40)
HH, TT = np.meshgrid(H, T)
Rf_surf = kern_seaton_Rf(TT, HH, 150.0, 1.5)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(HH, TT / 1000, Rf_surf * 1e4, cmap='plasma', alpha=0.9, edgecolor='none')
ax.set_xlabel('Hardness [ppm]')
ax.set_ylabel('Time [x1000 h]')
ax.set_zlabel('Rf x 10^4 [m2K/W]')
ax.set_title('3D Fouling Surface -- Kern-Seaton Model')
fig.colorbar(surf, ax=ax, shrink=0.6, label='Rf x 10^4')
ax.view_init(elev=22, azim=135)
plt.tight_layout()
plt.savefig('05_3d_fouling_surface.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 05_3d_fouling_surface.png')


## 3. Synthetic training data + physics-inspired predictor

Generate operational samples from Kern-Seaton, then fit a ridge-style linear model on physics-inspired features (numpy only).


In [ ]:
n = 2500
hours = np.random.uniform(0, 40000, n)
m_hot = np.random.uniform(1.5, 4.0, n)
T_hot = np.random.uniform(80, 180, n)
hardness = np.random.uniform(50, 600, n)
pH = np.random.uniform(6.5, 8.5, n)
velocity = np.random.uniform(0.5, 2.5, n)
noise = np.random.normal(0, 5e-6, n)
Rf = np.clip(kern_seaton_Rf(hours, hardness, T_hot, velocity) + noise, 0, None)

train_df = pd.DataFrame({
    'operating_hours': hours,
    'm_dot_hot': m_hot,
    'T_hot_in': T_hot,
    'water_hardness': hardness,
    'pH': pH,
    'velocity': velocity,
    'Rf': Rf,
})
train_df.to_csv('fouling_training_data.csv', index=False)
print(train_df.describe().round(4).to_string())
print('\nWrote fouling_training_data.csv (%d rows)' % len(train_df))


In [ ]:
def design_matrix(frame):
    '''Physics-inspired features aligned with Kern-Seaton drivers.'''
    h = frame['operating_hours'].values.astype(float)
    hard = frame['water_hardness'].values.astype(float)
    Th = frame['T_hot_in'].values.astype(float)
    vel = frame['velocity'].values.astype(float)
    return np.column_stack([
        np.ones(len(frame)),
        h / 10000.0,
        hard / 200.0,
        np.exp(0.01 * (Th - 100.0)),
        1.0 / np.maximum(vel, 0.3),
        (hard / 200.0) * (h / 10000.0),
        np.log1p(h),
    ])


idx = np.random.permutation(len(train_df))
split = int(0.8 * len(train_df))
tr, te = idx[:split], idx[split:]
Xtr, ytr = design_matrix(train_df.iloc[tr]), train_df['Rf'].values[tr]
Xte, yte = design_matrix(train_df.iloc[te]), train_df['Rf'].values[te]

lam = 1e-6
beta = np.linalg.solve(Xtr.T @ Xtr + lam * np.eye(Xtr.shape[1]), Xtr.T @ ytr)
y_pred = np.clip(Xte @ beta, 0, None)
r2 = 1 - np.sum((yte - y_pred) ** 2) / np.sum((yte - yte.mean()) ** 2)
mae = np.mean(np.abs(yte - y_pred))
print('Model R2  = %.4f' % r2)
print('Model MAE = %.2e m2K/W' % mae)


def predict_Rf(hours_v, hardness_v, T_v, velocity_v):
    hours_v = np.atleast_1d(np.asarray(hours_v, dtype=float))
    nloc = len(hours_v)
    def as_arr(v):
        return np.full(nloc, float(v)) if np.isscalar(v) or np.ndim(v) == 0 else np.asarray(v, dtype=float)
    tmp = pd.DataFrame({
        'operating_hours': hours_v,
        'water_hardness': as_arr(hardness_v),
        'T_hot_in': as_arr(T_v),
        'velocity': as_arr(velocity_v),
    })
    return np.clip(design_matrix(tmp) @ beta, 0, None)


In [ ]:
corr_imp = {
    c: abs(np.corrcoef(train_df[c], train_df['Rf'])[0, 1])
    for c in ['operating_hours', 'm_dot_hot', 'T_hot_in', 'water_hardness', 'pH', 'velocity']
}
imp_df = pd.Series(corr_imp).sort_values()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].scatter(yte, y_pred, alpha=0.45, s=18, c='royalblue', edgecolors='k', lw=0.3)
lims = [0, max(yte.max(), y_pred.max()) * 1.05]
axes[0].plot(lims, lims, 'r--', lw=2, label='Perfect')
axes[0].set_xlabel('Actual Rf [m2K/W]'); axes[0].set_ylabel('Predicted Rf')
axes[0].set_title('Parity Plot (R2 = %.3f)' % r2); axes[0].legend()

axes[1].barh(imp_df.index, imp_df.values, color='coral', edgecolor='k')
axes[1].set_xlabel('|Correlation| with Rf'); axes[1].set_title('Feature Influence')

hrs = np.linspace(0, 40000, 120)
for h in [100, 300, 500]:
    axes[2].plot(hrs, predict_Rf(hrs, h, 150, 1.5), lw=2, label='%d ppm' % h)
axes[2].set_xlabel('Operating Hours'); axes[2].set_ylabel('Predicted Rf')
axes[2].set_title('Fouling Growth vs Hardness'); axes[2].legend()
plt.tight_layout()
plt.savefig('01_ml_fouling_prediction.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 01_ml_fouling_prediction.png')


## 4. Plant-like sensor time series (180 days)

Simulate daily Rf, U, heat duty, efficiency and pressure drop under progressive fouling.


In [ ]:
n_days = 180
dates = pd.date_range('2026-01-01', periods=n_days, freq='D')
hours_ts = np.arange(n_days) * 24.0
hardness_ts = 280 + 40 * np.sin(2 * np.pi * np.arange(n_days) / 90)
velocity_ts = np.clip(1.4 + 0.15 * np.random.randn(n_days), 0.8, 2.2)
T_hot_ts = 145 + 8 * np.sin(2 * np.pi * np.arange(n_days) / 60) + np.random.randn(n_days)

Rf_ts = np.clip(
    kern_seaton_Rf(hours_ts, hardness_ts, T_hot_ts, velocity_ts) + np.random.normal(0, 3e-6, n_days),
    0, None
)
U_clean, Q_clean = 950.0, 680.0
U_ts = 1.0 / (1.0 / U_clean + Rf_ts)
Q_ts = Q_clean * (U_ts / U_clean) * (0.98 + 0.02 * np.random.randn(n_days))
eff_ts = Q_ts / Q_clean * 100
dP_ts = 28.0 * (1 + 180 * Rf_ts + 4.5e5 * Rf_ts ** 2) + np.random.normal(0, 0.6, n_days)

sensor = pd.DataFrame({
    'date': dates,
    'operating_hours': hours_ts,
    'water_hardness': hardness_ts,
    'velocity': velocity_ts,
    'T_hot_in': T_hot_ts,
    'Rf': Rf_ts,
    'U': U_ts,
    'Q_kW': Q_ts,
    'efficiency_pct': eff_ts,
    'delta_P_kPa': dP_ts,
})
print(sensor[['date', 'Rf', 'U', 'efficiency_pct', 'delta_P_kPa']].head(3).to_string(index=False))
print('...')
print(sensor[['date', 'Rf', 'U', 'efficiency_pct', 'delta_P_kPa']].tail(3).to_string(index=False))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes[0, 0].plot(sensor['date'], sensor['Rf'] * 1e4, 'b-', lw=1.5)
axes[0, 0].axhline(Rf_CRIT * 1e4, color='r', ls='--', label='Critical')
axes[0, 0].set_ylabel('Rf x 10^4'); axes[0, 0].set_title('Fouling Resistance'); axes[0, 0].legend()
axes[0, 1].plot(sensor['date'], sensor['efficiency_pct'], 'g-', lw=1.5)
axes[0, 1].set_ylabel('Efficiency %'); axes[0, 1].set_title('Thermal Efficiency')
axes[1, 0].plot(sensor['date'], sensor['delta_P_kPa'], 'm-', lw=1.5)
axes[1, 0].axhline(dP_CRIT, color='r', ls='--')
axes[1, 0].set_ylabel('dP [kPa]'); axes[1, 0].set_title('Pressure Drop')
axes[1, 1].plot(sensor['date'], sensor['U'], 'c-', lw=1.5)
axes[1, 1].set_ylabel('U [W/m2K]'); axes[1, 1].set_title('Overall Heat Transfer Coefficient')
for ax in axes.flat:
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('02_degradation_tracking.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 02_degradation_tracking.png')


## 5. Early anomaly detection

Flag days with abnormal Rf rate-of-change or dP residual spikes (z-score style).


In [ ]:
dRf = np.r_[0, np.diff(sensor['Rf'].values)]
z_dRf = (dRf - dRf.mean()) / (dRf.std() + 1e-12)
roll_mean = sensor['delta_P_kPa'].rolling(14, min_periods=3).mean()
roll_std = sensor['delta_P_kPa'].rolling(14, min_periods=3).std()
z_dP = (sensor['delta_P_kPa'] - roll_mean) / (roll_std + 1e-12)

sensor['anomaly'] = np.where((np.abs(z_dRf) > 2.5) | (np.abs(z_dP.fillna(0)) > 2.5), -1, 1)
sensor['anomaly_score'] = -np.maximum(np.abs(z_dRf), np.abs(z_dP.fillna(0).values))

n_anom = int((sensor['anomaly'] == -1).sum())
print('Detected %d anomalous days (%.1f%% of record)' % (n_anom, 100 * n_anom / len(sensor)))
if n_anom:
    print(sensor.loc[sensor['anomaly'] == -1, ['date', 'Rf', 'delta_P_kPa', 'anomaly_score']].tail(5).to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(sensor['date'], sensor['Rf'] * 1e4, 'b-', lw=1.2, label='Rf')
anom = sensor[sensor['anomaly'] == -1]
ax.scatter(anom['date'], anom['Rf'] * 1e4, c='red', s=40, zorder=5, label='Anomaly')
ax.axhline(Rf_CRIT * 1e4, color='r', ls='--', alpha=0.6)
ax.set_ylabel('Rf x 10^4')
ax.set_title('Early Fouling Detection (jump / residual anomalies)')
ax.legend()
plt.tight_layout()
plt.savefig('04_anomaly_detection.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 04_anomaly_detection.png')


## 6. Health index (0-100) & Remaining Useful Life


In [ ]:
sensor['health_index'] = (
    0.40 * np.clip(1 - sensor['Rf'] / Rf_CRIT, 0, 1) +
    0.25 * np.clip(1 - sensor['delta_P_kPa'] / dP_CRIT, 0, 1) +
    0.25 * np.clip((sensor['efficiency_pct'] - 70) / 30, 0, 1) +
    0.10 * np.clip(1 + sensor['anomaly_score'] / 5, 0, 1)
) * 100

recent = sensor.tail(45)
slope = np.polyfit(recent['operating_hours'], recent['Rf'], 1)[0]
last_Rf = float(sensor['Rf'].iloc[-1])
days_to_crit = max(0.0, (Rf_CRIT - last_Rf) / slope / 24.0) if slope > 1e-12 else 999.0

hi_now = float(sensor['health_index'].iloc[-1])
print('Current Health Index : %.1f / 100' % hi_now)
print('Current Rf           : %.6f m2K/W (critical %.1e)' % (last_Rf, Rf_CRIT))
print('Estimated RUL        : ~ %.0f days' % days_to_crit)
if days_to_crit < 7:
    print('>>> IMMEDIATE cleaning recommended')
elif days_to_crit < 30:
    print('>>> Plan cleaning in next maintenance window')
else:
    print('>>> No urgent action required')

sensor.to_csv('fouling_sensor_timeseries.csv', index=False)
print('Wrote fouling_sensor_timeseries.csv')


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(sensor['date'], sensor['health_index'], alpha=0.4, color='steelblue')
ax.plot(sensor['date'], sensor['health_index'], color='navy', lw=1.5)
ax.axhline(80, color='g', ls='--', label='Healthy >= 80')
ax.axhline(60, color='orange', ls='--', label='Watch < 60')
ax.axhline(40, color='r', ls='--', label='Action < 40')
ax.set_ylabel('Health Index')
ax.set_title('Fouling-Aware Health Index (0-100)')
ax.legend(loc='lower left')
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig('03_health_index.png', dpi=150, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(8, 2.8))
color = 'seagreen' if days_to_crit > 30 else 'orange' if days_to_crit > 10 else 'crimson'
ax.barh(['RUL'], [days_to_crit], color=color, edgecolor='k')
ax.axvline(30, color='orange', ls='--', label='30 d')
ax.axvline(7, color='r', ls='--', label='7 d')
ax.set_xlabel('Days until critical Rf')
ax.set_title('Remaining Useful Life ~ %.0f days' % days_to_crit)
ax.legend()
ax.set_xlim(0, max(120, days_to_crit * 1.2))
plt.tight_layout()
plt.savefig('08_rul_estimate.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 03_health_index.png and 08_rul_estimate.png')


## 7. Cleaning interval economics


In [ ]:
ENERGY_PRICE = 0.12   # $/kWh
CLEAN_COST = 4500     # $ per cleaning
DOWNTIME_COST = 18 * 85  # hours * $/h production loss

intervals = [15, 30, 45, 60, 90, 120]
rows = []
for iv in intervals:
    n_clean = 365 / iv
    avg_loss_frac = 0.5 * min(0.22, iv / 200)
    energy = avg_loss_frac * Q_clean * 24 * 365 * ENERGY_PRICE
    clean_tot = n_clean * (CLEAN_COST + DOWNTIME_COST)
    total = energy + clean_tot
    rows.append({'interval_d': iv, 'cleans_per_yr': n_clean, 'energy_loss_$': energy,
                 'clean_cost_$': clean_tot, 'total_$': total})

cost_df = pd.DataFrame(rows)
print(cost_df.round(0).to_string(index=False))
best = cost_df.loc[cost_df['total_$'].idxmin()]
print('\nSuggested interval under these assumptions: %d days' % int(best['interval_d']))

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['seagreen' if t == cost_df['total_$'].min() else 'steelblue' for t in cost_df['total_$']]
ax.bar(cost_df['interval_d'].astype(str), cost_df['total_$'], color=colors, edgecolor='k')
ax.set_xlabel('Cleaning interval [days]')
ax.set_ylabel('Annual cost [USD]')
ax.set_title('Cleaning Interval Optimization (energy loss + cleaning)')
plt.tight_layout()
plt.savefig('06_cleaning_cost_optimization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 06_cleaning_cost_optimization.png')


## 8. Summary table & dashboard


In [ ]:
summary = pd.DataFrame({
    'Metric': [
        'Model R2', 'Model MAE (m2K/W)', 'Current Rf', 'Critical Rf',
        'Health Index', 'RUL (days)', 'Anomalies detected', 'Final efficiency %',
        'Suggested clean interval (d)'
    ],
    'Value': [
        '%.4f' % r2, '%.2e' % mae, '%.2e' % last_Rf, '%.2e' % Rf_CRIT,
        '%.1f' % hi_now, '%.0f' % days_to_crit, '%d' % n_anom,
        '%.1f' % sensor['efficiency_pct'].iloc[-1], '%d' % int(best['interval_d'])
    ]
})
summary.to_csv('fouling_detection_summary.csv', index=False)
print('=' * 60)
print('FOULING DETECTION -- EXECUTIVE SUMMARY')
print('=' * 60)
print(summary.to_string(index=False))
print('=' * 60)
print('Diagrams: 01..08 PNG files | Data: training + sensor CSVs | Summary CSV written')


## How to use with real plant data

1. Export daily (or hourly) historian data with columns similar to `fouling_sensor_timeseries.csv`.
2. Estimate Rf from clean vs current overall U: `Rf ≈ 1/U_fouled - 1/U_clean`.
3. Set your site `Rf_CRIT`, `dP_CRIT`, energy price and cleaning cost.
4. Re-run anomaly, health-index and RUL cells.
5. Use the cleaning-interval chart to plan maintenance windows.


## 9. More features (v1.1)

| Feature | Description |
|---------|-------------|
| Multi-horizon Rf forecast | 0–90 day projection with uncertainty band |
| What-if scenarios | Hardness / velocity / temperature impact on fouling |
| Fleet comparison | Side-by-side Rf & health for multiple HX assets |
| RUL uncertainty | Bootstrap distribution of remaining useful life |
| Alert log | CSV of HI / Rf / anomaly events |
| KPI dashboard | Single-page summary figure |


### 9.1 Multi-horizon Rf forecast


In [ ]:
# Quadratic trend on recent window + residual band
recent = sensor.tail(60)
hrs = recent['operating_hours'].values
rf = recent['Rf'].values
coef = np.polyfit(hrs, rf, 2)
future_days = np.arange(0, 91)
last_h = hrs[-1]
fut_h = last_h + future_days * 24
rf_f = np.clip(np.polyval(coef, fut_h), 0, None)
sig = (rf - np.polyval(coef, hrs)).std()
rf_lo = np.clip(rf_f - 1.64 * sig, 0, None)
rf_hi = rf_f + 1.64 * sig

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(sensor['date'], sensor['Rf'] * 1e4, 'b-', lw=1.2, label='History')
fut_dates = pd.date_range(sensor['date'].iloc[-1], periods=91, freq='D')
ax.plot(fut_dates, rf_f * 1e4, 'r-', lw=2, label='Forecast')
ax.fill_between(fut_dates, rf_lo * 1e4, rf_hi * 1e4, color='red', alpha=0.2, label='90% band')
ax.axhline(Rf_CRIT * 1e4, color='k', ls='--', label='Critical Rf')
ax.axvline(sensor['date'].iloc[-1], color='gray', ls=':')
ax.set_ylabel('Rf x 10^4'); ax.set_title('Multi-horizon Rf forecast (0-90 days)'); ax.legend(loc='upper left')
plt.tight_layout(); plt.savefig('09_rf_forecast.png', dpi=150, bbox_inches='tight'); plt.show()
print('Saved 09_rf_forecast.png')


### 9.2 What-if scenarios (hardness / velocity / temperature)


In [ ]:
base_hard, base_T, base_v = 280, 145, 1.4
scenarios = {
    'Baseline': (base_hard, base_T, base_v),
    'Hardness +30%': (base_hard * 1.3, base_T, base_v),
    'Velocity +20%': (base_hard, base_T, base_v * 1.2),
    'T_hot +10 C': (base_hard, base_T + 10, base_v),
    'Soft water -40%': (base_hard * 0.6, base_T, base_v),
}
t_w = np.linspace(0, 20000, 100)
fig, ax = plt.subplots(figsize=(10, 4.5))
for name, (h, T, v) in scenarios.items():
    ax.plot(t_w / 1000, kern_seaton_Rf(t_w, h, T, v) * 1e4, lw=2, label=name)
ax.axhline(Rf_CRIT * 1e4, color='k', ls='--', alpha=0.7)
ax.set_xlabel('Operating time [x1000 h]'); ax.set_ylabel('Rf x 10^4')
ax.set_title('What-if: process changes vs fouling growth'); ax.legend()
plt.tight_layout(); plt.savefig('10_whatif_scenarios.png', dpi=150, bbox_inches='tight'); plt.show()
print('Saved 10_whatif_scenarios.png')


### 9.3 Fleet fouling comparison


In [ ]:
fleet_rows = []
for asset, hard0, vel0 in [('HX-A', 250, 1.5), ('HX-B', 320, 1.2), ('HX-C', 180, 1.8)]:
    n = 180
    hrs_a = np.arange(n) * 24.0
    hard = hard0 + 30 * np.sin(2 * np.pi * np.arange(n) / 80)
    vel = np.clip(vel0 + 0.1 * np.random.randn(n), 0.7, 2.3)
    T = 140 + 5 * np.random.randn(n)
    Rf_a = np.clip(kern_seaton_Rf(hrs_a, hard, T, vel) + np.random.normal(0, 3e-6, n), 0, None)
    U = 1.0 / (1.0 / 950 + Rf_a)
    HI = 100 * np.clip(1 - Rf_a / Rf_CRIT, 0, 1) * 0.7 + 30 * (U / 950)
    fleet_rows.append(pd.DataFrame({'asset': asset, 'day': np.arange(n), 'Rf': Rf_a, 'HI': HI}))
fleet_df = pd.concat(fleet_rows, ignore_index=True)
fleet_df.to_csv('fouling_fleet_comparison.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for asset, g in fleet_df.groupby('asset'):
    axes[0].plot(g['day'], g['Rf'] * 1e4, lw=1.5, label=asset)
    axes[1].plot(g['day'], g['HI'], lw=1.5, label=asset)
axes[0].axhline(Rf_CRIT * 1e4, color='r', ls='--')
axes[0].set_title('Fleet Rf'); axes[0].set_xlabel('Day'); axes[0].legend()
axes[1].axhline(60, color='orange', ls='--')
axes[1].set_title('Fleet Health Index'); axes[1].set_xlabel('Day'); axes[1].legend()
plt.tight_layout(); plt.savefig('11_fleet_fouling_comparison.png', dpi=150, bbox_inches='tight'); plt.show()
print('Saved 11_fleet_fouling_comparison.png | fouling_fleet_comparison.csv')


### 9.4 RUL uncertainty (bootstrap)


In [ ]:
n_boot = 200
rul_samples = []
win = sensor.tail(45)
for _ in range(n_boot):
    sample = win.sample(len(win), replace=True)
    s = np.polyfit(sample['operating_hours'], sample['Rf'], 1)[0]
    last = float(sensor['Rf'].iloc[-1])
    rul_samples.append(max(0, (Rf_CRIT - last) / s / 24) if s > 1e-12 else 999)
rul_samples = np.array(rul_samples)
rul_med = np.median(rul_samples)
rul_lo, rul_hi = np.percentile(rul_samples, [10, 90])
print('RUL median = %.0f days | P10 = %.0f | P90 = %.0f' % (rul_med, rul_lo, rul_hi))

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(rul_samples[rul_samples < 500], bins=25, color='steelblue', edgecolor='k', alpha=0.85)
ax.axvline(rul_med, color='k', lw=2, label='Median %.0f d' % rul_med)
ax.axvline(rul_lo, color='orange', ls='--', label='P10 %.0f d' % rul_lo)
ax.axvline(rul_hi, color='orange', ls='--', label='P90 %.0f d' % rul_hi)
ax.set_xlabel('RUL [days]'); ax.set_title('RUL uncertainty (bootstrap)'); ax.legend()
plt.tight_layout(); plt.savefig('12_rul_uncertainty.png', dpi=150, bbox_inches='tight'); plt.show()
print('Saved 12_rul_uncertainty.png')


### 9.5 Alert log export


In [ ]:
alerts = []
for _, row in sensor.iterrows():
    reasons = []
    hi = row.get('health_index', 100)
    if hi < 40: reasons.append('HI_CRITICAL')
    elif hi < 60: reasons.append('HI_WATCH')
    if row['Rf'] >= Rf_CRIT: reasons.append('RF_CRITICAL')
    if row.get('anomaly', 1) == -1: reasons.append('ANOMALY')
    if reasons:
        alerts.append({'date': row['date'], 'Rf': row['Rf'], 'HI': hi, 'alerts': '|'.join(reasons)})
alert_df = pd.DataFrame(alerts)
alert_df.to_csv('fouling_alert_log.csv', index=False)
print('Alert events: %d' % len(alert_df))
if len(alert_df):
    print(alert_df.tail(8).to_string(index=False))
print('Wrote fouling_alert_log.csv')


### 9.6 KPI dashboard figure


In [ ]:
fig = plt.figure(figsize=(12, 6))
gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.3)
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(sensor['date'], sensor['Rf'] * 1e4, 'b-', lw=1)
ax1.axhline(Rf_CRIT * 1e4, color='r', ls='--'); ax1.set_title('Rf'); ax1.tick_params(axis='x', rotation=30)
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(sensor['date'], sensor['health_index'], 'navy', lw=1)
ax2.axhline(60, color='orange', ls='--'); ax2.set_title('Health Index'); ax2.tick_params(axis='x', rotation=30)
ax3 = fig.add_subplot(gs[0, 2])
ax3.barh(['RUL med'], [rul_med], color='seagreen', edgecolor='k')
ax3.errorbar(rul_med, 0, xerr=[[rul_med - rul_lo], [rul_hi - rul_med]], fmt='none', ecolor='k', capsize=6)
ax3.set_title('RUL ~ %.0f d' % rul_med); ax3.set_xlabel('Days')
ax4 = fig.add_subplot(gs[1, :])
if len(alert_df):
    tmp = alert_df.copy()
    tmp['month'] = pd.to_datetime(tmp['date']).dt.to_period('M').astype(str)
    cnt = tmp.groupby('month').size()
    ax4.bar(cnt.index, cnt.values, color='coral', edgecolor='k')
    ax4.set_title('Alert count by month'); ax4.set_ylabel('Alerts')
fig.suptitle('Fouling Detection Dashboard', fontweight='bold', fontsize=14)
plt.savefig('13_fouling_dashboard.png', dpi=150, bbox_inches='tight'); plt.show()
print('Saved 13_fouling_dashboard.png')


## 10. Digital Twin concepts for heat-exchanger fouling

A **digital twin** is a living virtual counterpart of a physical asset. For a heat exchanger it typically includes:

| Layer | Role in this suite |
|-------|--------------------|
| **Physical asset** | Shell-and-tube (or plate) HX with sensors (T, flow, ΔP, inferred Rf) |
| **Data / connectivity** | Historian CSV / daily samples (`fouling_sensor_timeseries.csv`) |
| **Models** | Kern-Seaton fouling, physics-inspired predictor, health index, RUL |
| **State estimation** | Align twin Rf with measurements (assimilation / residual monitoring) |
| **Services** | Forecast, what-if, fleet compare, anomaly alerts, cleaning economics |
| **Decisions** | Clean now / plan window / watch — closed loop back to operations |

```mermaid
flowchart TB
    subgraph Physical
        HX[Heat Exchanger]
        SEN[Sensors: T, m, dP, U]
        HX --> SEN
    end

    subgraph Twin["Digital Twin"]
        DATA[Data layer]
        MOD[Kern-Seaton + Predictor]
        STATE[Twin state: Rf, HI, RUL]
        SVC[Forecast · What-if · Alerts]
        DATA --> MOD --> STATE --> SVC
    end

    subgraph Act
        OPS[Operations / Maintenance]
        CLEAN[Cleaning execution]
    end

    SEN -->|historian / CSV| DATA
    SVC -->|alerts & plan| OPS
    OPS --> CLEAN --> HX
    CLEAN -->|reset Rf| STATE
```


### Twin state object

A minimal twin holds geometry/process context, current fouling state, health, and methods to update from a new sensor row and to forecast.


In [ ]:
class HeatExchangerDigitalTwin:
    '''Minimal digital twin for fouling-aware HX monitoring.'''

    def __init__(self, name='HX-1', U_clean=950.0, Rf_crit=1.8e-4, assimilate_alpha=0.35):
        self.name = name
        self.U_clean = U_clean
        self.Rf_crit = Rf_crit
        self.alpha = assimilate_alpha  # measurement blend factor
        self.Rf = 0.0
        self.Rf_model = 0.0
        self.hours = 0.0
        self.health_index = 100.0
        self.history = []

    def model_Rf(self, hours, hardness, T_hot, velocity):
        return float(kern_seaton_Rf(hours, hardness, T_hot, velocity))

    def update(self, hours, hardness, T_hot, velocity, Rf_measured=None, dP=None, efficiency=None):
        '''Advance twin with a new operating point; optionally assimilate measured Rf.'''
        self.hours = float(hours)
        self.Rf_model = self.model_Rf(hours, hardness, T_hot, velocity)
        if Rf_measured is not None and np.isfinite(Rf_measured):
            self.Rf = (1 - self.alpha) * self.Rf_model + self.alpha * float(Rf_measured)
        else:
            self.Rf = self.Rf_model
        # simple health from Rf (extend with dP/efficiency if provided)
        hi = 100 * np.clip(1 - self.Rf / self.Rf_crit, 0, 1)
        if efficiency is not None:
            hi = 0.7 * hi + 0.3 * np.clip((efficiency - 70) / 30, 0, 1) * 100
        self.health_index = float(hi)
        self.history.append({
            'hours': self.hours, 'Rf': self.Rf, 'Rf_model': self.Rf_model,
            'health_index': self.health_index
        })
        return self.snapshot()

    def snapshot(self):
        return {
            'name': self.name,
            'hours': self.hours,
            'Rf': self.Rf,
            'Rf_model': self.Rf_model,
            'U_est': 1.0 / (1.0 / self.U_clean + self.Rf) if self.Rf is not None else self.U_clean,
            'health_index': self.health_index,
            'status': self.status()
        }

    def status(self):
        if self.Rf >= self.Rf_crit or self.health_index < 40:
            return 'CRITICAL'
        if self.health_index < 60:
            return 'WATCH'
        return 'HEALTHY'

    def forecast_Rf(self, horizon_hours, hardness, T_hot, velocity):
        return self.model_Rf(self.hours + horizon_hours, hardness, T_hot, velocity)


# Demo: drive twin from sensor CSV
twin = HeatExchangerDigitalTwin(name='ShellTube-A')
snaps = []
for _, row in sensor.iterrows():
    snap = twin.update(
        hours=row['operating_hours'],
        hardness=row['water_hardness'],
        T_hot=row['T_hot_in'],
        velocity=row['velocity'],
        Rf_measured=row['Rf'],
        efficiency=row.get('efficiency_pct', None)
    )
    snaps.append(snap)

print('Final twin snapshot:')
for k, v in snaps[-1].items():
    print('  %s: %s' % (k, v if not isinstance(v, float) else round(v, 6)))
print('History points:', len(twin.history))


In [ ]:
# Physical vs open-loop twin vs assimilated twin state
twin_Rf = np.array([h['Rf_model'] for h in twin.history])
assim_Rf = np.array([h['Rf'] for h in twin.history])
meas_Rf = sensor['Rf'].values

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(sensor['date'], meas_Rf * 1e4, 'b-', lw=1.5, label='Physical (measured)')
axes[0].plot(sensor['date'], twin_Rf * 1e4, 'r--', lw=1.5, label='Open-loop twin')
axes[0].plot(sensor['date'], assim_Rf * 1e4, 'g-', lw=1.8, label='Assimilated twin state')
axes[0].axhline(Rf_CRIT * 1e4, color='k', ls=':', alpha=0.6)
axes[0].set_ylabel('Rf x 10^4')
axes[0].set_title('Digital Twin vs Physical Asset')
axes[0].legend(loc='upper left')

residual = meas_Rf - twin_Rf
axes[1].fill_between(sensor['date'], residual * 1e4, 0, where=residual >= 0, color='tomato', alpha=0.5, label='Meas > Twin')
axes[1].fill_between(sensor['date'], residual * 1e4, 0, where=residual < 0, color='steelblue', alpha=0.5, label='Meas < Twin')
axes[1].axhline(0, color='k', lw=0.8)
axes[1].set_ylabel('Residual x 10^4')
axes[1].set_title('Model residual (unmodeled effects / calibration drift)')
axes[1].legend(loc='upper right')
plt.tight_layout()
plt.savefig('14_digital_twin_alignment.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 14_digital_twin_alignment.png')


### Closed loop: twin → decision → physical asset

```mermaid
sequenceDiagram
    participant P as Physical HX
    participant T as Digital Twin
    participant O as Operator / CMMS
    P->>T: Daily sensors (T, m, dP, U)
    T->>T: Update Rf state + HI + RUL
    T->>O: Alert / clean recommendation
    O->>P: Cleaning (if approved)
    P->>T: Post-clean Rf ~ 0 (reset)
```

The same twin object can serve **what-if** (change hardness/velocity), **fleet** (one twin per asset), and **forecast** (horizon_hours) without changing the physical unit until a maintenance action is taken.


In [ ]:
# Example: twin-based what-if from current state
print('What-if from current twin hours = %.0f' % twin.hours)
for label, hard, vel in [
    ('Hold current', 280, 1.4),
    ('Harder water +30%', 280 * 1.3, 1.4),
    ('Increase velocity +20%', 280, 1.4 * 1.2),
]:
    rf30 = twin.forecast_Rf(30 * 24, hard, 145, vel)
    rf90 = twin.forecast_Rf(90 * 24, hard, 145, vel)
    print('  %-28s  Rf@30d=%.2e  Rf@90d=%.2e' % (label, rf30, rf90))


## 11. More features (v1.2)

| Feature | Description |
|---------|-------------|
| Deposit thickness | Estimate layer thickness from Rf and deposit conductivity |
| Residual drift | Flag periods where twin residual mean exceeds 2σ |
| Cleaning verification | Simulate clean event and Rf regrowth |
| Threshold sensitivity | Cost index vs Rf alarm threshold |
| Sensor quality flags | OK / STUCK / OUT_OF_RANGE checks |
| Twin JSON snapshot | Machine-readable state for dashboards / APIs |


### 11.1 Deposit thickness estimate


In [ ]:
# Rf = x / k_deposit  =>  x = Rf * k
k_dep = 1.2  # W/mK (typical scale / deposit range ~0.5-2.5)
thickness_um = sensor['Rf'].values * k_dep * 1e6
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(sensor['date'], thickness_um, color='purple', lw=1.5)
ax.set_ylabel('Estimated thickness [μm]')
ax.set_title('Deposit thickness from Rf (k_deposit = %.1f W/mK)' % k_dep)
plt.tight_layout(); plt.savefig('16_deposit_thickness.png', dpi=150, bbox_inches='tight'); plt.show()
print('Current estimated thickness: %.1f μm' % thickness_um[-1])


### 11.2 Twin residual drift detection


In [ ]:
twin_ol = kern_seaton_Rf(sensor['operating_hours'], sensor['water_hardness'],
                         sensor['T_hot_in'], sensor['velocity'])
resid = sensor['Rf'].values - twin_ol
win = 14
roll = pd.Series(resid).rolling(win, min_periods=5).mean().values
drift_flag = np.abs(roll) > (2.0 * np.nanstd(resid))

fig, axes = plt.subplots(2, 1, figsize=(12, 5.5), sharex=True)
axes[0].plot(sensor['date'], resid * 1e4, 'steelblue', lw=1, label='Residual')
axes[0].plot(sensor['date'], roll * 1e4, 'r-', lw=2, label='%d-day mean' % win)
axes[0].axhline(0, color='k', lw=0.8)
axes[0].set_ylabel('Residual x 10^4'); axes[0].legend(); axes[0].set_title('Twin residual & drift monitor')
axes[1].fill_between(sensor['date'], 0, 1, where=drift_flag, color='tomato', alpha=0.4, label='Drift flag')
axes[1].set_ylabel('Flag'); axes[1].set_ylim(-0.1, 1.3); axes[1].legend()
axes[1].set_title('Drift when |rolling residual| > 2 global σ')
plt.tight_layout(); plt.savefig('17_residual_drift.png', dpi=150, bbox_inches='tight'); plt.show()
print('Drift-flagged days: %d / %d' % (int(drift_flag.sum()), len(sensor)))


### 11.3 Cleaning event verification


In [ ]:
Rf_sim = sensor['Rf'].values.copy()
clean_day = 120
post_h = np.arange(len(sensor) - clean_day) * 24.0
Rf_post = kern_seaton_Rf(post_h, sensor['water_hardness'].values[clean_day:],
                         sensor['T_hot_in'].values[clean_day:], sensor['velocity'].values[clean_day:])
Rf_sim[clean_day:] = Rf_post * 0.95
Rf_sim[clean_day] = 5e-6

fig, ax = plt.subplots(figsize=(11, 4.2))
ax.plot(sensor['date'], sensor['Rf'] * 1e4, 'b-', alpha=0.5, label='No clean')
ax.plot(sensor['date'], Rf_sim * 1e4, 'g-', lw=2, label='Clean at day 120')
ax.axvline(sensor['date'].iloc[clean_day], color='r', ls='--', label='Cleaning event')
ax.axhline(Rf_CRIT * 1e4, color='k', ls=':', alpha=0.6)
ax.set_ylabel('Rf x 10^4'); ax.set_title('Cleaning event verification (twin scenario)'); ax.legend()
plt.tight_layout(); plt.savefig('18_cleaning_event_verification.png', dpi=150, bbox_inches='tight'); plt.show()
print('Saved 18_cleaning_event_verification.png')


### 11.4 Rf threshold sensitivity


In [ ]:
thresholds = np.linspace(1.0e-4, 2.5e-4, 16)
ENERGY_PRICE, Q_clean, CLEAN_COST = 0.12, 680.0, 4500
costs = []
for thr in thresholds:
    days_over = int((sensor['Rf'] >= thr).sum())
    n_cleans = max(1, days_over // 30)
    loss_frac = np.clip(sensor['Rf'].mean() / thr * 0.08, 0.02, 0.25)
    energy = loss_frac * Q_clean * 24 * 180 * ENERGY_PRICE
    costs.append(energy + n_cleans * CLEAN_COST)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresholds * 1e4, costs, 'o-', color='darkslateblue', lw=2)
ax.axvline(Rf_CRIT * 1e4, color='r', ls='--', label='Current Rf_CRIT')
ax.set_xlabel('Rf threshold x 10^4'); ax.set_ylabel('Relative cost index (180 d)')
ax.set_title('Cost sensitivity to Rf alarm threshold'); ax.legend()
plt.tight_layout(); plt.savefig('19_threshold_sensitivity.png', dpi=150, bbox_inches='tight'); plt.show()


### 11.5 Sensor quality flags + twin JSON snapshot


In [ ]:
sq = sensor.copy()
sq['qf_Rf'] = 'OK'
sq.loc[sq['Rf'] < 0, 'qf_Rf'] = 'INVALID'
sq.loc[sq['Rf'] > 5e-4, 'qf_Rf'] = 'OUT_OF_RANGE'
roll_std_rf = sq['Rf'].rolling(5, min_periods=3).std()
sq.loc[roll_std_rf < 1e-9, 'qf_Rf'] = 'STUCK'
sq['qf_velocity'] = np.where((sq['velocity'] < 0.3) | (sq['velocity'] > 3.5), 'OUT_OF_RANGE', 'OK')
sq[['date', 'Rf', 'velocity', 'qf_Rf', 'qf_velocity']].to_csv('sensor_quality_flags.csv', index=False)
print(sq['qf_Rf'].value_counts().to_string())
print('Wrote sensor_quality_flags.csv')

from datetime import datetime
snap = {
    'generated': datetime.utcnow().isoformat() + 'Z',
    'asset': 'ShellTube-A',
    'Rf_current': float(sensor['Rf'].iloc[-1]),
    'Rf_crit': float(Rf_CRIT),
    'deposit_thickness_um': float(thickness_um[-1]),
    'drift_days_flagged': int(drift_flag.sum()),
    'status': 'WATCH' if sensor['Rf'].iloc[-1] > 0.7 * Rf_CRIT else 'HEALTHY'
}
with open('twin_snapshot.json', 'w') as f:
    json.dump(snap, f, indent=2)
print('Wrote twin_snapshot.json')
print(json.dumps(snap, indent=2))


## 12. More features (v1.3)

| Feature | Description |
|---------|-------------|
| Fouling rate | dRf/dt with 7-day smoothing |
| Correlation heatmap | Process variables vs Rf |
| Thickness uncertainty | Band from k_deposit range |
| Clean recovery | U/Q restoration after simulated clean |
| Alarm priority | P1/P2/P3 timeline |
| Work order JSON | Maintenance request stub |


### 12.1 Fouling rate dRf/dt


In [ ]:
dRf = np.r_[np.nan, np.diff(sensor['Rf'].values)]
dRf_s = pd.Series(dRf).rolling(7, min_periods=3).mean().values
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(sensor['date'], dRf * 1e6, color='gray', alpha=0.4, label='Daily dRf')
ax.plot(sensor['date'], dRf_s * 1e6, color='darkorange', lw=2, label='7-day mean')
ax.axhline(0, color='k', lw=0.8)
ax.set_ylabel('dRf/dt [x1e-6 m2K/W per day]')
ax.set_title('Fouling rate monitoring'); ax.legend()
plt.tight_layout(); plt.savefig('20_fouling_rate.png', dpi=150, bbox_inches='tight'); plt.show()


### 12.2 Correlation heatmap


In [ ]:
cols = [c for c in ['operating_hours','water_hardness','velocity','T_hot_in','Rf','U','efficiency_pct','delta_P_kPa'] if c in sensor.columns]
corr = sensor[cols].corr()
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(cols))); ax.set_yticks(range(len(cols)))
ax.set_xticklabels(cols, rotation=45, ha='right'); ax.set_yticklabels(cols)
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j, i, '%.2f' % corr.values[i, j], ha='center', va='center', fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title('Correlation heatmap')
plt.tight_layout(); plt.savefig('21_correlation_heatmap.png', dpi=150, bbox_inches='tight'); plt.show()


### 12.3 Thickness uncertainty & clean recovery


In [ ]:
k_lo, k_hi, k_mid = 0.6, 2.0, 1.2
fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(sensor['date'], sensor['Rf']*k_lo*1e6, sensor['Rf']*k_hi*1e6, color='purple', alpha=0.25, label='k=0.6-2.0')
ax.plot(sensor['date'], sensor['Rf']*k_mid*1e6, color='purple', lw=2, label='k=1.2')
ax.set_ylabel('Thickness [μm]'); ax.set_title('Deposit thickness uncertainty'); ax.legend()
plt.tight_layout(); plt.savefig('22_thickness_uncertainty.png', dpi=150, bbox_inches='tight'); plt.show()

U_clean = 950.0
clean_day = 120
U_before = 1.0 / (1.0 / U_clean + sensor['Rf'].iloc[clean_day - 1])
U_after = 1.0 / (1.0 / U_clean + 5e-6)
recovery_pct = (U_after - U_before) / (U_clean - U_before) * 100
print('U recovery after clean: %.1f%% of lost conductance restored' % recovery_pct)


### 12.4 Alarm priority & work order


In [ ]:
hi = sensor['health_index'].values if 'health_index' in sensor.columns else 100 * np.clip(1 - sensor['Rf']/Rf_CRIT, 0, 1)
anom = sensor['anomaly'].values if 'anomaly' in sensor.columns else np.ones(len(sensor))
prio = np.ones(len(sensor), dtype=int)
prio[hi < 60] = 2; prio[hi < 40] = 3
prio[anom == -1] = np.maximum(prio[anom == -1], 2)
prio[sensor['Rf'].values >= Rf_CRIT] = 3

fig, ax = plt.subplots(figsize=(11, 3.5))
for p, c, lab in [(1,'seagreen','P1 Low'), (2,'orange','P2 Medium'), (3,'crimson','P3 High')]:
    m = prio == p
    ax.scatter(sensor['date'][m], prio[m], c=c, s=28, label=lab, zorder=3)
ax.set_yticks([1,2,3]); ax.set_ylim(0.5, 3.5)
ax.set_ylabel('Priority'); ax.set_title('Alarm priority timeline'); ax.legend(loc='upper left')
plt.tight_layout(); plt.savefig('24_alarm_priority.png', dpi=150, bbox_inches='tight'); plt.show()

wo = {
    'work_order_id': 'WO-HX-FOUL-001',
    'asset': 'ShellTube-A',
    'created': datetime.utcnow().isoformat() + 'Z',
    'priority': 'P2',
    'reason': 'Health index in WATCH band; fouling rate elevated',
    'recommended_action': 'Schedule chemical clean within 30 days',
    'current_Rf': float(sensor['Rf'].iloc[-1]),
    'estimated_cost_usd': 4500 + 18 * 85
}
with open('maintenance_work_order.json', 'w') as f:
    json.dump(wo, f, indent=2)
print('Wrote maintenance_work_order.json')
print(json.dumps(wo, indent=2))


## 13. More features (v1.4)

| Feature | Description |
|---------|-------------|
| Partial response curves | Rf vs hardness / velocity / T (other inputs fixed) |
| Cumulative energy loss | kWh and $ lost to fouling over the record |
| Traffic-light status | GREEN / YELLOW / RED bands + 30-day counts |
| Velocity trade-off | Fouling reduction vs pumping power |
| Weekly text report | `weekly_fouling_report.txt` for email / CMMS |


### 13.1 Partial response curves


In [ ]:
base = dict(hours=8000, hardness=280, T_hot=145, velocity=1.4)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
h_grid = np.linspace(80, 550, 50)
axes[0].plot(h_grid, kern_seaton_Rf(base['hours'], h_grid, base['T_hot'], base['velocity']) * 1e4, 'b-', lw=2)
axes[0].axhline(Rf_CRIT * 1e4, color='r', ls='--')
axes[0].set_xlabel('Hardness [ppm]'); axes[0].set_ylabel('Rf x 10^4'); axes[0].set_title('vs hardness')
v_grid = np.linspace(0.5, 2.5, 50)
axes[1].plot(v_grid, kern_seaton_Rf(base['hours'], base['hardness'], base['T_hot'], v_grid) * 1e4, 'g-', lw=2)
axes[1].axhline(Rf_CRIT * 1e4, color='r', ls='--')
axes[1].set_xlabel('Velocity [m/s]'); axes[1].set_title('vs velocity')
T_grid = np.linspace(80, 180, 50)
axes[2].plot(T_grid, kern_seaton_Rf(base['hours'], base['hardness'], T_grid, base['velocity']) * 1e4, 'm-', lw=2)
axes[2].axhline(Rf_CRIT * 1e4, color='r', ls='--')
axes[2].set_xlabel('T_hot [C]'); axes[2].set_title('vs hot temperature')
plt.suptitle('Partial response curves (baseline elsewhere)', fontweight='bold')
plt.tight_layout(); plt.savefig('26_partial_response_curves.png', dpi=150, bbox_inches='tight'); plt.show()


### 13.2 Cumulative energy loss & traffic light


In [ ]:
U_clean, Q_clean, price = 950.0, 680.0, 0.12
U = 1.0 / (1.0 / U_clean + sensor['Rf'].values)
Q = Q_clean * (U / U_clean)
cum_loss_kWh = np.cumsum((Q_clean - Q) * 24)
cum_cost = cum_loss_kWh * price

fig, ax1 = plt.subplots(figsize=(11, 4.2))
ax1.fill_between(sensor['date'], cum_loss_kWh, alpha=0.35, color='crimson')
ax1.plot(sensor['date'], cum_loss_kWh, color='crimson', lw=1.5)
ax1.set_ylabel('Cumulative energy loss [kWh]', color='crimson')
ax2 = ax1.twinx()
ax2.plot(sensor['date'], cum_cost, color='navy', lw=1.5, ls='--')
ax2.set_ylabel('Cumulative cost [USD]', color='navy')
ax1.set_title('Cumulative energy & cost loss from fouling')
plt.tight_layout(); plt.savefig('27_cumulative_energy_loss.png', dpi=150, bbox_inches='tight'); plt.show()
print('Total loss: %.0f kWh | $%.0f' % (cum_loss_kWh[-1], cum_cost[-1]))

hi = sensor['health_index'].values if 'health_index' in sensor.columns else 100 * np.clip(1 - sensor['Rf']/Rf_CRIT, 0, 1)
status = np.where(hi >= 80, 'GREEN', np.where(hi >= 60, 'YELLOW', 'RED'))
counts = pd.Series(status[-30:]).value_counts().reindex(['GREEN','YELLOW','RED']).fillna(0)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(sensor['date'], hi, color='navy', lw=1.2)
axes[0].axhline(80, color='g', ls='--'); axes[0].axhline(60, color='orange', ls='--')
axes[0].set_ylim(0, 105); axes[0].set_title('Traffic-light health bands')
axes[1].bar(counts.index, counts.values, color=['seagreen','orange','crimson'], edgecolor='k')
axes[1].set_title('Last 30 days status count')
plt.tight_layout(); plt.savefig('28_traffic_light_status.png', dpi=150, bbox_inches='tight'); plt.show()


### 13.3 Velocity trade-off & weekly report


In [ ]:
vels = np.linspace(0.6, 2.4, 40)
Rf_at = kern_seaton_Rf(10000, 300, 150, vels)
pump_proxy = (vels / 1.4) ** 3
fig, ax1 = plt.subplots(figsize=(9, 4.2))
ax1.plot(vels, Rf_at * 1e4, 'b-o', ms=3, lw=2)
ax1.set_xlabel('Velocity [m/s]'); ax1.set_ylabel('Rf x 10^4', color='b')
ax2 = ax1.twinx()
ax2.plot(vels, pump_proxy, 'r--', lw=2)
ax2.set_ylabel('Relative pumping power', color='r')
ax1.set_title('Velocity: fouling vs pumping power')
ax1.axhline(Rf_CRIT * 1e4, color='k', ls=':', alpha=0.6)
plt.tight_layout(); plt.savefig('29_velocity_tradeoff.png', dpi=150, bbox_inches='tight'); plt.show()

from datetime import datetime
last7 = sensor.tail(7)
lines = [
    '=' * 60,
    'WEEKLY FOULING TWIN REPORT',
    'Generated: ' + datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC'),
    '=' * 60,
    'Rf end: %.3e m2K/W (crit %.3e)' % (last7['Rf'].iloc[-1], Rf_CRIT),
    'U mean: %.1f W/m2K | Eff: %.1f%%' % (last7['U'].mean(), last7['efficiency_pct'].mean()),
    'Status: %s' % status[-1],
    '=' * 60,
]
open('weekly_fouling_report.txt', 'w').write('\n'.join(lines) + '\n')
print('\n'.join(lines))
print('Wrote weekly_fouling_report.txt')


## 14. More features (v1.5)

| Feature | Description |
|---------|-------------|
| Change-point CUSUM | Detect shifts in fouling rate |
| Forecast backtest | MAE at 7 / 14 / 30 day horizons |
| Maintenance calendar | Suggested future clean dates from RUL |
| RUL sensitivity tornado | Impact of Rf, slope, Rf_crit uncertainty |
| TEMA fouling factors | Typical design Rf by service |
| Dual-unit Rf sample | SI and imperial side by side |


### 14.1 Change-point detection (CUSUM)


In [ ]:
rf = sensor['Rf'].values
dRf = np.r_[0, np.diff(rf)]
mu, sig = dRf.mean(), dRf.std() + 1e-15
k, h = 0.5 * sig, 4 * sig
S = np.zeros(len(dRf)); cps = []
for i in range(1, len(dRf)):
    S[i] = max(0, S[i-1] + (dRf[i] - mu - k))
    if S[i] > h:
        cps.append(i); S[i] = 0

fig, axes = plt.subplots(2, 1, figsize=(12, 5.5), sharex=True)
axes[0].plot(sensor['date'], rf * 1e4, 'b-', lw=1.2)
for i in cps:
    axes[0].axvline(sensor['date'].iloc[i], color='r', alpha=0.35)
axes[0].axhline(Rf_CRIT * 1e4, color='k', ls='--', alpha=0.5)
axes[0].set_ylabel('Rf x 10^4'); axes[0].set_title('Rf with CUSUM change-points')
axes[1].plot(sensor['date'], S, 'purple', lw=1.2)
axes[1].axhline(h, color='r', ls='--', label='Threshold')
axes[1].set_ylabel('CUSUM S'); axes[1].legend()
plt.tight_layout(); plt.savefig('30_changepoint_cusum.png', dpi=150, bbox_inches='tight'); plt.show()
print('Change-points detected:', len(cps))


### 14.2 Forecast backtest & maintenance calendar


In [ ]:
horizons = [7, 14, 30]
errors = {h: [] for h in horizons}
for t in range(70, len(sensor) - 31, 5):
    win = sensor.iloc[t-60:t]
    coef = np.polyfit(win['operating_hours'].values, win['Rf'].values, 2)
    h0 = sensor['operating_hours'].iloc[t-1]
    for H in horizons:
        pred = np.polyval(coef, h0 + H * 24)
        errors[H].append(abs(pred - sensor['Rf'].iloc[t + H - 1]))
mae = {h: np.mean(errors[h]) for h in horizons}
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([str(h) for h in horizons], [mae[h]*1e4 for h in horizons], color='teal', edgecolor='k')
ax.set_xlabel('Horizon [days]'); ax.set_ylabel('MAE Rf x 10^4')
ax.set_title('Forecast backtest MAE')
plt.tight_layout(); plt.savefig('31_forecast_backtest.png', dpi=150, bbox_inches='tight'); plt.show()
print('MAE:', {h: '%.2e' % mae[h] for h in horizons})

from datetime import timedelta
slope = np.polyfit(sensor['operating_hours'].tail(45), sensor['Rf'].tail(45), 1)[0]
last_Rf = float(sensor['Rf'].iloc[-1])
days_to_crit = max(1, (Rf_CRIT - last_Rf) / slope / 24) if slope > 1e-15 else 180
interval = int(np.clip(days_to_crit * 0.85, 30, 120))
start = sensor['date'].iloc[-1]
dates_clean = [start + timedelta(days=int(interval*(i+1))) for i in range(4)]
pd.DataFrame({'clean_index': range(1,5), 'suggested_date': dates_clean, 'interval_days': interval}).to_csv('maintenance_calendar.csv', index=False)
print('Suggested clean interval: %d days' % interval)
for i, d in enumerate(dates_clean, 1):
    print('  Clean %d: %s' % (i, d.date()))


### 14.3 RUL sensitivity tornado & TEMA factors


In [ ]:
def rul_days(Rf_last, slope_val, Rf_crit=Rf_CRIT):
    if slope_val <= 1e-15: return 999.0
    return max(0, (Rf_crit - Rf_last) / slope_val / 24)

base_rul = rul_days(last_Rf, slope)
factors = {
    'Rf_last +10%': rul_days(last_Rf*1.1, slope) - base_rul,
    'Rf_last -10%': rul_days(last_Rf*0.9, slope) - base_rul,
    'slope +20%': rul_days(last_Rf, slope*1.2) - base_rul,
    'slope -20%': rul_days(last_Rf, slope*0.8) - base_rul,
    'Rf_crit +10%': rul_days(last_Rf, slope, Rf_CRIT*1.1) - base_rul,
    'Rf_crit -10%': rul_days(last_Rf, slope, Rf_CRIT*0.9) - base_rul,
}
items = sorted(factors.items(), key=lambda kv: abs(kv[1]))
labels = [k for k,_ in items]
fig, ax = plt.subplots(figsize=(9, 4.5))
y = np.arange(len(labels))
ax.barh(y, [min(0,v) for _,v in items], color='coral', edgecolor='k', label='Negative')
ax.barh(y, [max(0,v) for _,v in items], color='seagreen', edgecolor='k', label='Positive')
ax.axvline(0, color='k', lw=0.8)
ax.set_yticks(y); ax.set_yticklabels(labels)
ax.set_xlabel('Change in RUL [days]')
ax.set_title('RUL sensitivity tornado (base ~ %.0f d)' % base_rul)
ax.legend(loc='lower right')
plt.tight_layout(); plt.savefig('33_rul_sensitivity_tornado.png', dpi=150, bbox_inches='tight'); plt.show()

tema = pd.DataFrame({
    'Service': ['Sea water ≤50C','Sea water >50C','Cooling tower treated','Cooling tower untreated',
                'River water','Fuel oil light','Fuel oil heavy','Crude oil','Steam (clean)','Air / process gas'],
    'Rf_m2K_W': [9e-5, 1.8e-4, 1.8e-4, 3.5e-4, 3.5e-4, 3.5e-4, 8.8e-4, 3.5e-4, 9e-5, 3.5e-4]
})
tema['Rf_ft2hF_Btu'] = tema['Rf_m2K_W'] * 5.678
tema.to_csv('tema_fouling_factors.csv', index=False)
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(tema['Service'], tema['Rf_m2K_W']*1e4, color='steelblue', edgecolor='k')
ax.axvline(Rf_CRIT*1e4, color='r', ls='--', label='Our Rf_CRIT')
ax.set_xlabel('Rf x 10^4 [m2K/W]'); ax.set_title('TEMA-style typical fouling factors'); ax.legend()
plt.tight_layout(); plt.savefig('34_tema_fouling_factors.png', dpi=150, bbox_inches='tight'); plt.show()
print(tema.to_string(index=False))


## 15. More features (v1.6)

| Feature | Description |
|---------|-------------|
| Twin fidelity R² | Rolling 30-day agreement of model vs measured Rf |
| Mitigation levers | Ranked impact of hardness / velocity / T changes |
| Duty loss waterfall | Design Q → fouling loss → current Q |
| Accelerated fouling RUL | RUL under faster growth scenarios |
| HTML dashboard | Lightweight `fouling_dashboard.html` status page |


### 15.1 Twin fidelity & mitigation levers


In [ ]:
twin = kern_seaton_Rf(sensor['operating_hours'], sensor['water_hardness'],
                      sensor['T_hot_in'], sensor['velocity'])
meas = sensor['Rf'].values
win = 30
r2_roll = []
for i in range(len(sensor)):
    if i < win:
        r2_roll.append(np.nan); continue
    yt, yp = meas[i-win:i], twin[i-win:i]
    r2_roll.append(1 - np.sum((yt-yp)**2) / (np.sum((yt-yt.mean())**2) + 1e-18))
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(sensor['date'], r2_roll, 'b-', lw=1.5)
ax.axhline(0.8, color='g', ls='--', label='Good ≥ 0.8')
ax.axhline(0.5, color='orange', ls='--', label='Weak < 0.5')
ax.set_ylabel('Rolling R² (30 d)'); ax.set_title('Digital twin fidelity'); ax.legend()
plt.tight_layout(); plt.savefig('35_twin_fidelity_r2.png', dpi=150, bbox_inches='tight'); plt.show()

h0 = float(sensor['operating_hours'].iloc[-1])
hard0 = float(sensor['water_hardness'].iloc[-1])
T0 = float(sensor['T_hot_in'].iloc[-1]); v0 = float(sensor['velocity'].iloc[-1])
base_f = float(kern_seaton_Rf(h0+60*24, hard0, T0, v0))
levers = {
    'Hardness -20%': float(kern_seaton_Rf(h0+60*24, hard0*0.8, T0, v0)) - base_f,
    'Hardness -40%': float(kern_seaton_Rf(h0+60*24, hard0*0.6, T0, v0)) - base_f,
    'Velocity +15%': float(kern_seaton_Rf(h0+60*24, hard0, T0, v0*1.15)) - base_f,
    'Velocity +30%': float(kern_seaton_Rf(h0+60*24, hard0, T0, v0*1.3)) - base_f,
    'T_hot -10 C': float(kern_seaton_Rf(h0+60*24, hard0, T0-10, v0)) - base_f,
    'T_hot -20 C': float(kern_seaton_Rf(h0+60*24, hard0, T0-20, v0)) - base_f,
}
lev = pd.Series(levers).sort_values()
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(lev.index, lev.values*1e4, color=['seagreen' if v<0 else 'coral' for v in lev.values], edgecolor='k')
ax.axvline(0, color='k', lw=0.8)
ax.set_xlabel('ΔRf @ 60d x 10^4'); ax.set_title('Mitigation lever ranking')
plt.tight_layout(); plt.savefig('36_mitigation_levers.png', dpi=150, bbox_inches='tight'); plt.show()
print(lev)


### 15.2 Duty waterfall, accelerated RUL, HTML dashboard


In [ ]:
U_clean, Q_clean = 950.0, 680.0
U_now = 1.0 / (1.0 / U_clean + meas[-1])
Q_now = Q_clean * (U_now / U_clean)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(['Design Q'], [Q_clean], color='steelblue', edgecolor='k')
ax.bar(['Fouling loss'], [Q_clean - Q_now], bottom=[Q_now], color='crimson', edgecolor='k')
ax.bar(['Current Q'], [Q_now], color='seagreen', edgecolor='k')
ax.set_ylabel('kW'); ax.set_title('Duty loss waterfall')
plt.tight_layout(); plt.savefig('37_duty_loss_waterfall.png', dpi=150, bbox_inches='tight'); plt.show()
print('Q design=%.0f kW | Q now=%.0f kW | loss=%.0f kW' % (Q_clean, Q_now, Q_clean-Q_now))

slope = np.polyfit(sensor['operating_hours'].tail(45), sensor['Rf'].tail(45), 1)[0]
last_Rf = float(meas[-1])
def rul(Rf, s):
    return max(0, (Rf_CRIT - Rf) / s / 24) if s > 1e-15 else 999
sc = {
    'Current trend': rul(last_Rf, slope),
    'Accelerated x1.5': rul(last_Rf, slope*1.5),
    'Accelerated x2': rul(last_Rf, slope*2),
}
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(list(sc.keys()), list(sc.values()), color=['seagreen','orange','crimson'], edgecolor='k')
ax.set_xlabel('RUL [days]'); ax.set_title('RUL under accelerated fouling')
plt.tight_layout(); plt.savefig('38_accelerated_fouling_rul.png', dpi=150, bbox_inches='tight'); plt.show()
print(sc)

hi = float(sensor['health_index'].iloc[-1]) if 'health_index' in sensor.columns else 50.0
status = 'GREEN' if hi >= 80 else ('YELLOW' if hi >= 60 else 'RED')
html = '''<!DOCTYPE html><html><head><meta charset="utf-8"><title>HX Fouling Twin</title>
<style>body{font-family:system-ui,sans-serif;margin:24px;background:#0f172a;color:#e2e8f0}
.card{background:#1e293b;border-radius:12px;padding:16px;margin-bottom:12px}
.RED{color:#f87171}.YELLOW{color:#fbbf24}.GREEN{color:#4ade80}
.kpi{display:inline-block;background:#334155;border-radius:8px;padding:12px;margin:6px;min-width:120px}
.v{font-size:1.4rem;font-weight:700}</style></head><body>
<h1>Fouling Digital Twin</h1>
<div class="card"><b>Status:</b> <span class="%s">%s</span>
<div class="kpi">Rf<div class="v">%.2e</div></div>
<div class="kpi">HI<div class="v">%.0f</div></div>
<div class="kpi">Q<div class="v">%.0f kW</div></div>
</div>
<p>See PNG charts in this folder for full diagnostics.</p>
</body></html>''' % (status, status, meas[-1], hi, Q_now)
open('fouling_dashboard.html','w').write(html)
print('Wrote fouling_dashboard.html | status=', status)


## 16. More features (v1.7)

| Feature | Description |
|---------|-------------|
| Residual diagnostics | Histogram + QQ-style residual plot |
| Cleaning ROI | Payback days from energy savings vs clean cost |
| Fleet clean priority | Rank assets by who to clean first |
| Time-to-threshold | Days to alarm vs chosen Rf_CRIT |
| Operator checklist | Markdown daily / weekly / clean workflow |


### 16.1 Residual diagnostics & cleaning ROI


In [ ]:
twin = kern_seaton_Rf(sensor['operating_hours'], sensor['water_hardness'],
                      sensor['T_hot_in'], sensor['velocity'])
resid = sensor['Rf'].values - twin
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(resid * 1e4, bins=30, color='steelblue', edgecolor='k', density=True)
axes[0].axvline(0, color='r', ls='--')
axes[0].set_xlabel('Residual x 10^4'); axes[0].set_title('Twin residual distribution')
sorted_r = np.sort(resid * 1e4)
prob = (np.arange(1, len(sorted_r)+1) - 0.5) / len(sorted_r)
theor = 0.55 * np.log(prob / (1 - prob))
axes[1].scatter(theor, sorted_r, s=12, alpha=0.6, c='purple')
axes[1].set_title('Residual QQ-style plot')
plt.tight_layout(); plt.savefig('39_residual_diagnostics.png', dpi=150, bbox_inches='tight'); plt.show()

U_clean, Q_clean, price = 950.0, 680.0, 0.12
CLEAN_COST = 4500 + 18 * 85
U_now = 1.0 / (1.0 / U_clean + sensor['Rf'].iloc[-1])
Q_now = Q_clean * (U_now / U_clean)
daily_gain_kWh = (Q_clean - Q_now) * 24
payback_days = CLEAN_COST / (daily_gain_kWh * price)
days = np.arange(1, 181)
fig, ax = plt.subplots(figsize=(9, 4.2))
ax.plot(days, days * daily_gain_kWh * price, 'g-', lw=2, label='Cumulative savings')
ax.axhline(CLEAN_COST, color='r', ls='--', label='Clean cost')
ax.axvline(payback_days, color='orange', ls=':', label='Payback ~ %.0f d' % payback_days)
ax.set_xlabel('Days after clean'); ax.set_ylabel('USD')
ax.set_title('Cleaning ROI'); ax.legend()
plt.tight_layout(); plt.savefig('40_clean_roi.png', dpi=150, bbox_inches='tight'); plt.show()
print('Payback days: %.0f' % payback_days)


### 16.2 Fleet priority, time-to-threshold, checklist


In [ ]:
fleet = []
for asset, hard0, vel0, hours0 in [
    ('HX-A', 250, 1.5, 4000), ('HX-B', 320, 1.2, 9000),
    ('HX-C', 180, 1.8, 2500), ('HX-D', 400, 1.0, 12000)]:
    Rf = float(kern_seaton_Rf(hours0, hard0, 150, vel0))
    loss = 680 - 680 * ((1.0 / (1.0/950 + Rf)) / 950)
    HI = 100 * np.clip(1 - Rf / Rf_CRIT, 0, 1)
    slope = float(kern_seaton_Rf(hours0 + 24*30, hard0, 150, vel0) - Rf) / 30
    rul = max(0, (Rf_CRIT - Rf) / (slope + 1e-15)) if slope > 0 else 999
    score = (1 - HI/100)*40 + (1 - min(rul, 180)/180)*40 + (loss/100)*20
    fleet.append(dict(asset=asset, Rf=Rf, HI=HI, Q_loss_kW=loss, RUL_d=rul, priority_score=score))
fleet_df = pd.DataFrame(fleet).sort_values('priority_score', ascending=False)
fleet_df.to_csv('fleet_clean_priority.csv', index=False)
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(fleet_df['asset'], fleet_df['priority_score'], color='darkslateblue', edgecolor='k')
ax.set_xlabel('Priority score'); ax.set_title('Fleet: who to clean first')
plt.tight_layout(); plt.savefig('41_fleet_clean_priority.png', dpi=150, bbox_inches='tight'); plt.show()
print(fleet_df.to_string(index=False))

slope = np.polyfit(sensor['operating_hours'].tail(45), sensor['Rf'].tail(45), 1)[0]
last_Rf = float(sensor['Rf'].iloc[-1])
crits = np.linspace(1.0e-4, 3.0e-4, 25)
tt = [max(0, (c - last_Rf)/slope/24) if slope > 1e-15 and c > last_Rf else 0 for c in crits]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(crits*1e4, tt, 'b-o', ms=4, lw=2)
ax.axvline(Rf_CRIT*1e4, color='r', ls='--', label='Current Rf_CRIT')
ax.set_xlabel('Rf threshold x 10^4'); ax.set_ylabel('Days to threshold')
ax.set_title('Time-to-threshold vs Rf_CRIT'); ax.legend()
plt.tight_layout(); plt.savefig('42_time_to_threshold.png', dpi=150, bbox_inches='tight'); plt.show()
print('Wrote operator_checklist.md and fleet_clean_priority.csv')
